# Load the trained models and define inference helpers


In [1]:
import os
from io import StringIO
import sys
import traceback

import joblib
import pandas as pd

from jpeg_compression_detection import detect_jpeg_compression
from jpeg_compression_detection.paths import DATA_DIR, MODELS_DIR, display_path

classification_model_path = MODELS_DIR / "best_classification_model.pkl"
scaler_path = MODELS_DIR / "scaler.pkl"
feature_columns_path = MODELS_DIR / "feature_columns.pkl"

loaded_model = loaded_scaler = loaded_feature_columns = None
required_classification_files = [classification_model_path, scaler_path, feature_columns_path]
if all(path.is_file() for path in required_classification_files):
    loaded_model = joblib.load(classification_model_path)
    loaded_scaler = joblib.load(scaler_path)
    loaded_feature_columns = joblib.load(feature_columns_path)
    print("Model loaded successfully!")
    print(f"Model type: {type(loaded_model).__name__}")
    print(f"Number of features: {len(loaded_feature_columns)}")
else:
    missing = [display_path(path) for path in required_classification_files if not path.is_file()]
    print("Classification setup skipped; missing model files: " + ", ".join(missing))


def check_image(indicators, verbose=True):
    """Classify an image as JFIF or PPM from its calculated indicators."""
    if indicators is None or not isinstance(indicators, dict):
        if verbose:
            print("Error: no valid indicators were supplied")
        return None
    if any(item is None for item in (loaded_model, loaded_scaler, loaded_feature_columns)):
        if verbose:
            print("Classification skipped because the model files are unavailable")
        return None

    if verbose:
        print("\n           JFIF vs PPM classification")
        print("-" * 50)

    try:
        feature_dict = {column: indicators.get(column, 0) for column in loaded_feature_columns}
        single_row = pd.DataFrame([feature_dict])
        if single_row.isnull().any().any():
            if verbose:
                print("\nMissing indicator values:")
                print(single_row.isnull().sum()[single_row.isnull().sum() > 0])
            return None

        scaled_row = loaded_scaler.transform(single_row)
        prediction = loaded_model.predict(scaled_row)[0]
        probabilities = loaded_model.predict_proba(scaled_row)[0]

        if verbose:
            format_name = "PPM (uncompressed)" if prediction == 1 else "JFIF (JPEG compression)"
            confidence = max(probabilities) * 100
            print(f"Detected format: {format_name}")
            print(f"Confidence: {confidence:.1f}%")

        return {
            "success": True,
            "prediction": prediction,
            "format": "PPM" if prediction == 1 else "JFIF",
            "probabilities": {"JFIF": probabilities[0], "PPM": probabilities[1]},
            "confidence": max(probabilities) * 100,
        }
    except Exception as exc:
        if verbose:
            print(f"Error: {exc}")
            traceback.print_exc()
        return None


print("Function check_image() updated!")


Model loaded successfully!
Model type: SVC
Number of features: 15
Function check_image() updated!


In [2]:
compression_model_path = MODELS_DIR / "best_compression_model.pkl"
loaded_compression_model = (
    joblib.load(compression_model_path)
    if compression_model_path.is_file()
    else None
)
if loaded_compression_model is None:
    print(f"Quality prediction skipped; model file is missing: {display_path(compression_model_path)}")


In [3]:
def predict_compression_quality(indicators):
    """Predict JPEG compression quality from calculated image indicators."""
    if indicators is None or not isinstance(indicators, dict):
        return {"success": False, "error": "No valid indicators were supplied"}
    if loaded_compression_model is None or loaded_feature_columns is None:
        return {"success": False, "error": "The quality model files are unavailable"}

    print("\n\n           Compression quality prediction")
    print("-" * 50)
    try:
        feature_dict = {
            column: indicators.get(column, 0)
            for column in loaded_feature_columns
            if column != "estimated_quality"
        }
        single_row = pd.DataFrame([feature_dict])
        if single_row.isnull().any().any():
            missing = single_row.isnull().sum()[single_row.isnull().sum() > 0]
            return {"success": False, "error": f"Missing indicator data: {missing.to_dict()}"}

        predicted_quality = loaded_compression_model.predict(single_row)[0]
        print(f"Predicted JPEG quality: {predicted_quality:.1f}")
        print("\n\nIndicators used by the model:")
        for key in loaded_feature_columns:
            if key in indicators and key != "estimated_quality":
                value = indicators[key]
                if isinstance(value, (int, float)):
                    print(f"  {key:30s}: {value:>10.4f}")

        return {"success": True, "predicted_quality": predicted_quality, "indicators": indicators}
    except Exception as exc:
        error_message = f"Error: {exc}\n{traceback.format_exc()}"
        print(error_message)
        return {"success": False, "error": error_message}


print("Function predict_compression_quality() updated!")


Function predict_compression_quality() updated!


# Predict PPM/JFIF format and estimate JPEG quality


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

directory_path = DATA_DIR / "contest-images"
text_font_size = 16
title_font_size = 16
image_extensions = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".tiff", ".tif", ".jfif")

print(f"Analyzing directory: {display_path(directory_path)}")
models_available = all(
    item is not None
    for item in (loaded_model, loaded_scaler, loaded_feature_columns, loaded_compression_model)
)

if not models_available:
    print("Analysis skipped because one or more model files are unavailable.")
elif not directory_path.is_dir():
    print(f"Analysis skipped; directory does not exist: {display_path(directory_path)}")
else:
    image_files = sorted(
        path for path in directory_path.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    if not image_files:
        print(f"No image files found in {display_path(directory_path)}")
    else:
        print(f"Found {len(image_files)} image files\n")
        for index, image_path in enumerate(image_files, 1):
            try:
                indicators = detect_jpeg_compression(image_path)
                if not isinstance(indicators, dict):
                    print(f"Failed to compute indicators for {image_path.name}: {indicators}")
                    continue

                old_stdout = sys.stdout
                text_buffer = StringIO()
                try:
                    sys.stdout = text_buffer
                    check_image(indicators, verbose=True)
                    predict_compression_quality(indicators)
                finally:
                    sys.stdout = old_stdout
                text_output = text_buffer.getvalue()

                with Image.open(image_path) as image:
                    figure, (image_axis, text_axis) = plt.subplots(1, 2, figsize=(18, 10))
                    image_axis.imshow(image)
                image_axis.axis("off")
                image_axis.set_title(f"Test image [{index}/{len(image_files)}]: {image_path.name}", fontsize=title_font_size, fontweight="bold", pad=65)
                text_axis.axis("off")
                text_axis.text(0.05, 0.5, text_output, transform=text_axis.transAxes, fontsize=text_font_size, verticalalignment="center", fontfamily="monospace", bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.3})
                text_axis.set_title("Analysis results", fontsize=title_font_size, fontweight="bold", pad=20)
                plt.subplots_adjust(top=0.95, bottom=0.05, left=0.05, right=0.95, wspace=0.1)
                plt.show()
            except Exception as exc:
                print(f"Error while analyzing {image_path.name}: {exc}")
                traceback.print_exc()
